# Practice fine-tuning run: finance and tax

This notebook walks through one complete post-training loop on a small open model, so the hackathon workflow feels familiar:

1. **Score the base model** on held-out finance and tax questions.
2. **Fine-tune** it with LoRA (low-rank adaptation), which trains a small add-on instead of the whole model.
3. **Score it again** and compare.

**Runtime:** Google Colab with a free T4 GPU (graphics processing unit). Choose *Runtime → Change runtime type → T4 GPU* before running. The whole notebook takes roughly 15–25 minutes.

**What to expect:** with only 22 training examples, this is a rehearsal, not a real improvement. The model will mostly learn the answer format and may overfit. At the hackathon you would use hundreds to thousands of examples.

**Files to upload when prompted:** `train.jsonl`, `eval.jsonl` (from the `data` folder) and `score.py` (from the `scoring` folder).

## 1. Install the libraries
Unsloth makes fine-tuning fast enough for a free GPU. This takes 2–4 minutes. If Colab asks you to restart the session afterwards, do so and continue from step 2.

In [1]:
!pip install -q unsloth

## 2. Upload the data and scoring script

In [3]:
import json, os, sys

try:
    from google.colab import files
    print("Select train.jsonl, eval.jsonl and score.py")
    files.upload()
except ImportError:
    print("Not on Colab: make sure train.jsonl, eval.jsonl and score.py are in this folder.")

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

train_rows = load_jsonl("train.jsonl")
eval_rows = load_jsonl("eval.jsonl")
sys.path.insert(0, ".")
from score import score, print_report

print(f"{len(train_rows)} training examples, {len(eval_rows)} evaluation examples")
print("Example question:", train_rows[0]["question"][:200])

Select train.jsonl, eval.jsonl and score.py


Saving score.py to score.py
22 training examples, 10 evaluation examples
Example question: A company has pre-tax income of $2,400,000.00. The state tax rate is 6% of pre-tax income, and state tax is deductible when computing federal taxable income. The federal rate is 21%. What is the total


## 3. Load a small model
`Qwen2.5-1.5B-Instruct` is small enough for a free GPU. At the hackathon, swap in whichever model the rules allow (for example a 4B or 8B model on a larger GPU).

`load_in_4bit=True` compresses the model so it fits in memory.

In [4]:
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct"
MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.9: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

## 4. Decide how questions and answers are formatted
Every example becomes a short chat: a system instruction, the question, and (for training) the worked answer ending in `Final answer: ...`. The scoring script reads that last line, so teaching the model this exact format is the first easy win.

In [11]:
SYSTEM_PROMPT = (
    "You are a careful finance and tax analyst. Work through the problem step by step, "
    "then end with a single line in the form 'Final answer: <answer>'. "
    "Give percentages with a % sign and dollar amounts as plain numbers."
)

def build_messages(row, include_answer):
    question = row["question"]
    if row.get("context"):
        question = row["context"] + "\n\n" + question
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    if include_answer:
        answer = str(row["answer"]) + ("%" if row["answer_type"] == "percent" else "")
        messages.append({"role": "assistant", "content": row["reasoning"] + "\nFinal answer: " + answer})
    return messages

print(tokenizer.apply_chat_template(build_messages(train_rows[0], True), tokenize=False))

<|im_start|>system
You are a careful finance and tax analyst. Work through the problem step by step, then end with a single line in the form 'Final answer: <answer>'. Give percentages with a % sign and dollar amounts as plain numbers.<|im_end|>
<|im_start|>user
A company has pre-tax income of $2,400,000.00. The state tax rate is 6% of pre-tax income, and state tax is deductible when computing federal taxable income. The federal rate is 21%. What is the total income tax (state plus federal), in dollars?<|im_end|>
<|im_start|>assistant
State tax = $2,400,000.00 × 6% = $144,000.00.
Federal taxable income = $2,400,000.00 − $144,000.00 = $2,256,000.00.
Federal tax = $2,256,000.00 × 21% = $473,760.00.
Total = $144,000.00 + $473,760.00 = $617,760.00.
Final answer: 617760<|im_end|>



## 5. Score the base model before training
This is your baseline. Write the number down: every change you make later is judged against it.

In [12]:
import torch

def generate_answers(rows, max_new_tokens=384):
    FastLanguageModel.for_inference(model)
    outputs = {}
    for row in rows:
        prompt = tokenizer.apply_chat_template(
            build_messages(row, include_answer=False),
            add_generation_prompt=True,
            return_tensors="pt",
        ).to("cuda")
        with torch.no_grad():
            out = model.generate(input_ids=prompt, max_new_tokens=max_new_tokens, do_sample=False)
        outputs[row["id"]] = tokenizer.decode(out[0][prompt.shape[1]:], skip_special_tokens=True)
    return outputs

base_outputs = generate_answers(eval_rows)
base_report = score(eval_rows, base_outputs)
print_report(base_report, "base model")

Both `max_new_tokens` (=384) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=384) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=384) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=384) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Score — base model
------------------
Overall: 0/10 = 0.0%
  accounting           0/2 = 0%
  corporate_finance    0/3 = 0%
  fund                 0/1 = 0%
  tax                  0/1 = 0%
  venture              0/3 = 0%
Responses without a 'Final answer:' line: 5


### Read a few mistakes
This is the most valuable habit at the hackathon: look at *why* the model is wrong before deciding what data to add.

In [14]:
for err in base_report["errors"][:3]:
    print(f"--- {err['id']} ({err['category']}) expected {err['expected']}, got {err['got']}")
    print(base_outputs[err["id"]][-600:])
    print()

--- EV-001 (tax) expected 935625, got 6.825e+06
Step 1: Calculate the state tax.
State tax = Pre-tax income * State tax rate
State tax = $3,750,000.00 * 0.05

Step 2: Subtract the state tax from the pre-tax income to get the net income before federal tax.
Net income before federal tax = Pre-tax income - State tax

Step 3: Calculate the federal tax on the net income.
Federal tax = Net income before federal tax * Federal tax rate
Federal tax = (Pre-tax income - State tax) * 0.21

Step 4: Add the state tax and the federal tax to find the total income tax.

Final answer: 6,825,000

--- EV-002 (corporate_finance) expected 20.11, got 17.67
formula:

\[ CAGR = \left( \frac{V_f}{V_i} \right)^{\frac{1}{n}} - 1 \]

Where:
- \( V_f \) is the final value ($20,000,000.00),
- \( V_i \) is the initial value ($8,000,000.00),
- \( n \) is the number of periods (years).

Plugging in the values:

\[ CAGR = \left( \frac{20,000,000.00}{8,000,000.00} \right)^{\frac{1}{5}} - 1 \]
\[ CAGR = \left( 2.5 \right)

## 6. Attach LoRA adapters and fine-tune
- `r=16` sets the size of the add-on. Bigger can learn more but trains slower.
- `max_steps=30` keeps the practice run short. With 22 examples and 8 examples per step, the model sees each example about 11 times, which is a lot, so watch for overfitting.
- `learning_rate=2e-4` is a common starting point for LoRA.

In [15]:
import inspect
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
FastLanguageModel.for_training(model)

train_ds = Dataset.from_list([
    {"text": tokenizer.apply_chat_template(build_messages(r, include_answer=True), tokenize=False)}
    for r in train_rows
])

config_args = dict(
    dataset_text_field="text",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=3,
    max_steps=30,
    learning_rate=2e-4,
    logging_steps=5,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    output_dir="outputs",
    report_to="none",
)
# The TRL library renamed some arguments across versions; this keeps the cell working on either.
config_params = inspect.signature(SFTConfig).parameters
config_args["max_seq_length" if "max_seq_length" in config_params else "max_length"] = MAX_SEQ_LEN
trainer_args = dict(model=model, train_dataset=train_ds, args=SFTConfig(**config_args))
if "processing_class" in inspect.signature(SFTTrainer.__init__).parameters:
    trainer_args["processing_class"] = tokenizer
else:
    trainer_args["tokenizer"] = tokenizer

trainer = SFTTrainer(**trainer_args)
stats = trainer.train()
print(f"Training took {stats.metrics['train_runtime']:.0f} seconds; final loss {stats.metrics['train_loss']:.3f}")

Unsloth 2026.10.9 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/22 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 22 | Num Epochs = 10 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Step,Training Loss
5,1.805483
10,1.144345
15,0.659309
20,0.482680
25,0.426069
30,0.401680


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-30/tokenizer_config.json.


Training took 61 seconds; final loss 0.820


## 7. Score the fine-tuned model
Same questions, same scoring. Compare against the baseline from step 5.

In [16]:
tuned_outputs = generate_answers(eval_rows)
tuned_report = score(eval_rows, tuned_outputs)
print_report(base_report, "base model")
print()
print_report(tuned_report, "fine-tuned")
print()
print(f"Change: {tuned_report['accuracy'] - base_report['accuracy']:+.1%}")

Both `max_new_tokens` (=384) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=384) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=384) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=384) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Score — base model
------------------
Overall: 0/10 = 0.0%
  accounting           0/2 = 0%
  corporate_finance    0/3 = 0%
  fund                 0/1 = 0%
  tax                  0/1 = 0%
  venture              0/3 = 0%
Responses without a 'Final answer:' line: 5

Score — fine-tuned
------------------
Overall: 1/10 = 10.0%
  accounting           1/2 = 50%
  corporate_finance    0/3 = 0%
  fund                 0/1 = 0%
  tax                  0/1 = 0%
  venture              0/3 = 0%

Change: +10.0%


## 8. Save the error log and the adapter
The error log opens in Google Sheets. Fill in the `failure_type` column to see which kinds of mistakes are most common.

In [18]:
from score import write_errors

write_errors(tuned_report, "errors_after_training.csv")
model.save_pretrained("finance_tax_lora")
tokenizer.save_pretrained("finance_tax_lora")

try:
    from google.colab import files
    files.download("errors_after_training.csv")
except ImportError:
    print("Saved errors_after_training.csv and the finance_tax_lora folder.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## What to try next
- **More data beats more training.** Add 50–100 examples for whichever category scored worst, then rerun steps 6–7.
- **Check for overfitting.** If the score drops after training, lower `max_steps` or the learning rate.
- **Try a bigger model.** Change `MODEL_NAME` to a larger instruct model if your GPU has room.
- **Keep eval questions out of training.** Never copy evaluation questions into the training file, or the score stops meaning anything.